# 02c — Model/Config Selection (Default Hyperparameters, Walk-Forward CV)

**Stage A** of the revised pipeline. Selects the single best (model, window, horizon,
covariate-group) configuration using **default hyperparameters** and expanding-window
walk-forward CV with an embargo, replacing the old single 80/20 split
(`legacy_pre_revision/02a_phase1_screening_v2.ipynb`, `02b_default_params.ipynb`).

Everything is evaluated with the same untuned `DEFAULT_CONFIGS` so the comparison across
the 4 models is apples-to-apples — nothing is tuned yet. The winning configuration is
handed to `02d_nested_tuning_cv.ipynb` (Stage B), which runs Optuna scoped to that one
configuration only.

Grid: 4 models × 21 covariate scenarios (1 baseline + 15 single + 5 group) × 2 windows ×
3 horizons × 5 CV folds = 2,520 fits.

## Output
- `stage_a_fold_results.csv` — one row per Model×Covariates×Window×Horizon×Fold
- `stage_a_summary.csv` — mean/std MAPE etc. per Model×Covariates×Window×Horizon
- `saved_models/winning_config.joblib` — argmin mean CV MAPE configuration


In [1]:
import gc
import glob
import traceback
from datetime import datetime

import joblib
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

import cv_lib as cv

df_merged = joblib.load(sorted(glob.glob("saved_models/df_merged_*.joblib"), reverse=True)[0])
print(f"Data: {df_merged.shape} | {df_merged['date'].min().date()} to {df_merged['date'].max().date()}")

N_FOLDS = 5
total_exp = len(cv.MODEL_NAMES) * len(cv.SCENARIO_COVARIATES) * len(cv.WINDOWS) * len(cv.HORIZONS)
print(f"Models: {len(cv.MODEL_NAMES)} | Scenarios: {len(cv.SCENARIO_COVARIATES)} | "
      f"Windows: {cv.WINDOWS} | Horizons: {cv.HORIZONS}")
print(f"Total combos: {total_exp}  x  {N_FOLDS} folds = {total_exp * N_FOLDS} fits")


Data: (2408, 17) | 2015-02-23 to 2025-01-31
Models: 4 | Scenarios: 21 | Windows: [20, 120] | Horizons: [1, 5, 20]
Total combos: 504  x  5 folds = 2520 fits


## Smoke test

Reduced slice before committing to the full 2,520-fit run: 1 model (LightGBM, fastest) ×
1 window × 1 horizon × 3 scenarios × 2 folds. Confirms no exceptions and measures real
per-fit wall time so the full-run estimate below isn't a guess.


In [2]:
import time

smoke_scenarios = ["Baseline", "Screening1", "GDP"]
t0 = time.time()
smoke_results = []
for scenario_name in smoke_scenarios:
    scenario_vars = cv.SCENARIO_COVARIATES[scenario_name]
    target_ts, cov_ts = cv.to_series(df_merged, "IHSG", scenario_vars if scenario_vars else None)
    n = len(target_ts)
    folds = cv.expanding_window_folds(n, n_folds=2, embargo=1)
    for f in folds:
        m = cv.run_fold("LightGBM", cv.DEFAULT_CONFIGS["LightGBM"], target_ts, cov_ts,
                         window=20, horizon=1,
                         train_end=f["train_end"], test_start=f["test_start"], test_end=f["test_end"])
        smoke_results.append({"Covariates": scenario_name, "Fold": f["fold"], **m})
        print(f"  {scenario_name:12s} fold {f['fold']}: MAPE={m['mape']:.4f}%")

elapsed = time.time() - t0
n_fits = len(smoke_results)
per_fit = elapsed / n_fits
print(f"\n{n_fits} fits in {elapsed:.1f}s -> {per_fit:.2f}s/fit (LightGBM, W20, H1)")
print(f"Rough full-grid estimate at this per-fit rate: {per_fit * total_exp * N_FOLDS / 60:.1f} min")
print("NOTE: this is a lower bound -- RandomForest/ExtraTrees at 300 trees and H20 "
      "(20x more historical_forecasts steps) will be substantially slower than this LightGBM/H1 sample.")


  Baseline     fold 0: MAPE=0.8806%


  Baseline     fold 1: MAPE=0.5823%


  Screening1   fold 0: MAPE=0.8348%


  Screening1   fold 1: MAPE=0.5601%


  GDP          fold 0: MAPE=0.8806%


  GDP          fold 1: MAPE=0.5952%

6 fits in 4.2s -> 0.70s/fit (LightGBM, W20, H1)
Rough full-grid estimate at this per-fit rate: 29.3 min
NOTE: this is a lower bound -- RandomForest/ExtraTrees at 300 trees and H20 (20x more historical_forecasts steps) will be substantially slower than this LightGBM/H1 sample.


## Full grid (checkpointed)

Runs all 2,520 fits, checkpointing `stage_a_fold_results.csv` after each model finishes
(4 checkpoints total) so the run is resumable if interrupted. Re-running this cell from
scratch will redo everything — if resuming after a partial run, load the existing CSV
and filter `cv.MODEL_NAMES` down to the models not yet completed before re-running.


In [3]:
import os

# Resume-aware: if a checkpoint from a prior (possibly interrupted) run exists,
# load it and skip combos already completed instead of recomputing from scratch.
# Checkpoints save after every covariate SCENARIO (not just after every model) --
# this environment appears to kill long-running background processes around the
# ~50 minute mark, and XGBoost alone can take longer than that, so per-model
# checkpointing was too coarse: an interrupted XGBoost run lost all its progress.
if os.path.exists("stage_a_fold_results.csv"):
    df_checkpoint = pd.read_csv("stage_a_fold_results.csv")
    results = df_checkpoint.to_dict("records")
    done_combos = set(df_checkpoint.groupby(["Model", "Covariates", "Window", "Horizon"]).size().index)
    print(f"Resuming from checkpoint: {len(results)} rows, {len(done_combos)} combos already done")
else:
    results = []
    done_combos = set()

failed = []
exp_num = 0
start_time = datetime.now()

print(f"Started: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total  : {total_exp} combos x {N_FOLDS} folds = {total_exp * N_FOLDS} fits")
print("=" * 90)

for model_name in cv.MODEL_NAMES:
    for scenario_name, scenario_vars in cv.SCENARIO_COVARIATES.items():
        scenario_had_new_work = False
        for window in cv.WINDOWS:
            for horizon in cv.HORIZONS:
                exp_num += 1
                combo_key = (model_name, scenario_name, window, horizon)
                tag = (f"[{exp_num:>4}/{total_exp}] "
                       f"{model_name:13s} | {scenario_name:22s} | W{window:>3}_H{horizon:>2}")
                if combo_key in done_combos:
                    print(f"{tag} ... skipped (already in checkpoint)")
                    continue
                scenario_had_new_work = True
                embargo = horizon
                print(tag, end=" ... ", flush=True)
                try:
                    target_ts, cov_ts = cv.to_series(df_merged, "IHSG", scenario_vars if scenario_vars else None)
                    n = len(target_ts)
                    folds = cv.expanding_window_folds(n, n_folds=N_FOLDS, embargo=embargo)
                    fold_mapes = []
                    for f in folds:
                        m = cv.run_fold(model_name, cv.DEFAULT_CONFIGS[model_name], target_ts, cov_ts,
                                         window, horizon, f["train_end"], f["test_start"], f["test_end"])
                        results.append({
                            "Model": model_name, "Covariates": scenario_name,
                            "Window": window, "Horizon": horizon, "Fold": f["fold"], **m,
                        })
                        fold_mapes.append(m["mape"])
                    mean_mape = sum(fold_mapes) / len(fold_mapes) if fold_mapes else float("nan")
                    print(f"mean MAPE={mean_mape:.4f}% ({len(folds)} folds)")
                except Exception as e:
                    failed.append({"tag": tag, "error": str(e)})
                    print(f"FAILED: {e}")
                    traceback.print_exc()
                finally:
                    gc.collect()
        if scenario_had_new_work and results:
            pd.DataFrame(results).to_csv("stage_a_fold_results.csv", index=False)
            print(f"  -> Checkpoint saved ({len(results)} rows) after model={model_name}, scenario={scenario_name}")

elapsed = datetime.now() - start_time
print("=" * 90)
print(f"Done in {elapsed} | {len(results)} fold-results OK, {len(failed)} combos failed")

df_stage_a = pd.DataFrame(results)
df_stage_a.to_csv("stage_a_fold_results.csv", index=False)
print(f"Saved: stage_a_fold_results.csv ({len(df_stage_a)} rows)")

if failed:
    df_failed = pd.DataFrame(failed)
    df_failed.to_csv("stage_a_failed.csv", index=False)
    print(f"Saved: stage_a_failed.csv ({len(df_failed)} rows) -- inspect before trusting the summary below")

Resuming from checkpoint: 2400 rows, 480 combos already done
Started: 2026-07-13 03:28:13
Total  : 504 combos x 5 folds = 2520 fits
[   1/504] RandomForest  | Baseline               | W 20_H 1 ... skipped (already in checkpoint)
[   2/504] RandomForest  | Baseline               | W 20_H 5 ... skipped (already in checkpoint)
[   3/504] RandomForest  | Baseline               | W 20_H20 ... skipped (already in checkpoint)
[   4/504] RandomForest  | Baseline               | W120_H 1 ... skipped (already in checkpoint)
[   5/504] RandomForest  | Baseline               | W120_H 5 ... skipped (already in checkpoint)
[   6/504] RandomForest  | Baseline               | W120_H20 ... skipped (already in checkpoint)
[   7/504] RandomForest  | BI_Rate                | W 20_H 1 ... skipped (already in checkpoint)
[   8/504] RandomForest  | BI_Rate                | W 20_H 5 ... skipped (already in checkpoint)
[   9/504] RandomForest  | BI_Rate                | W 20_H20 ... skipped (already in checkpo

mean MAPE=0.6839% (5 folds)
[ 482/504] LightGBM      | Screening2             | W 20_H 5 ... 

mean MAPE=1.1640% (5 folds)
[ 483/504] LightGBM      | Screening2             | W 20_H20 ... 

mean MAPE=2.3443% (5 folds)
[ 484/504] LightGBM      | Screening2             | W120_H 1 ... 

mean MAPE=0.6836% (5 folds)
[ 485/504] LightGBM      | Screening2             | W120_H 5 ... 

...

mean MAPE=1.1630% (5 folds)
[ 501/504] LightGBM      | All_Covariates         | W 20_H20 ... 

mean MAPE=2.4635% (5 folds)
[ 502/504] LightGBM      | All_Covariates         | W120_H 1 ... 

mean MAPE=0.6879% (5 folds)
[ 503/504] LightGBM      | All_Covariates         | W120_H 5 ... 

mean MAPE=1.1726% (5 folds)
[ 504/504] LightGBM      | All_Covariates         | W120_H20 ... 

mean MAPE=2.2942% (5 folds)
  -> Checkpoint saved (2520 rows) after model=LightGBM, scenario=All_Covariates
Done in 0:29:19.191923 | 2520 fold-results OK, 0 combos failed
Saved: stage_a_fold_results.csv (2520 rows)


## Summary and winning configuration

Assert every combo produced the expected fold count (catches `n` too small for the
chosen fractions/embargo at some window/horizon combo -- would silently under-report
otherwise). Aggregate mean/std MAPE per combo, pick the winner by lowest mean MAPE.


In [4]:
df_stage_a = pd.read_csv("stage_a_fold_results.csv")

fold_counts = df_stage_a.groupby(["Model", "Covariates", "Window", "Horizon"]).size()
short = fold_counts[fold_counts < N_FOLDS]
if len(short):
    print(f"WARNING: {len(short)} combos produced fewer than {N_FOLDS} folds:")
    print(short)
else:
    print(f"All combos produced exactly {N_FOLDS} folds.")

df_stage_a_summary = (
    df_stage_a.groupby(["Model", "Covariates", "Window", "Horizon"])
    .agg(mape_mean=("mape", "mean"), mape_std=("mape", "std"),
         da_mean=("da", "mean"), mase_mean=("mase", "mean"),
         n_folds=("Fold", "count"))
    .reset_index()
)
df_stage_a_summary.to_csv("stage_a_summary.csv", index=False)
print(f"Saved: stage_a_summary.csv ({len(df_stage_a_summary)} rows)")

winner = df_stage_a_summary.sort_values("mape_mean").iloc[0]
winning_config = {
    "model": winner["Model"],
    "covariates": winner["Covariates"],
    "cov_vars": cv.SCENARIO_COVARIATES[winner["Covariates"]],
    "window": int(winner["Window"]),
    "horizon": int(winner["Horizon"]),
    "default_mape_mean": float(winner["mape_mean"]),
    "default_mape_std": float(winner["mape_std"]),
}
joblib.dump(winning_config, "saved_models/winning_config.joblib")

print("\nWinning configuration (default hyperparameters, lowest mean CV MAPE):")
for k, v in winning_config.items():
    print(f"  {k}: {v}")

print("\nTop 10 by mean MAPE:")
display(df_stage_a_summary.sort_values("mape_mean").head(10))


All combos produced exactly 5 folds.
Saved: stage_a_summary.csv (504 rows)

Winning configuration (default hyperparameters, lowest mean CV MAPE):
  model: ExtraTrees
  covariates: Screening1
  cov_vars: ['Silver', 'WTI', 'Gold', 'STI', 'Coal', 'Tin', 'NPL_Ratio']
  window: 120
  horizon: 1
  default_mape_mean: 0.6359999999999999
  default_mape_std: 0.15013808644044985

Top 10 by mean MAPE:


,Model,Covariates,Window,Horizon,mape_mean,mape_std,da_mean,mase_mean,n_folds
87,ExtraTrees,Screening1,120,1,0.63600,0.150138,52.424,1.11206,5
6,ExtraTrees,All_Covariates,20,1,0.63624,0.150802,52.526,1.11224,5
9,ExtraTrees,All_Covariates,120,1,0.63624,0.149967,52.318,1.11238,5
3,ExtraTrees,All_Commodity_STI,120,1,0.63662,0.151693,51.444,1.11278,5
42,ExtraTrees,Copper,20,1,0.63670,0.150321,52.784,1.11296,5
15,ExtraTrees,All_Macro_no_UST,120,1,0.63676,0.150270,52.268,1.11310,5
99,ExtraTrees,Silver,120,1,0.63702,0.150686,53.248,1.11332,5
258,RandomForest,All_Covariates,20,1,0.63718,0.151115,53.556,1.11388,5
93,ExtraTrees,Screening2,120,1,0.63726,0.151084,52.680,1.11398,5
69,ExtraTrees,NPL_Ratio,120,1,0.63730,0.151541,51.650,1.11374,5
